# 04 - Gold
Cruza as partidas com o clima da cidade do mandante no dia do jogo e calcula as métricas.

| Tabela | Grão | Para que serve |
|---|---|---|
| `gold_partidas_clima` | partida | Dataset final: partida + clima + métricas |
| `gold_resumo_clima` | dimensão × faixa | Responde: o clima influencia gols e resultados? |

In [0]:
%load_ext autoreload
%autoreload 2

In [0]:
import logging
import os
import sys

sys.path.insert(0, os.path.abspath(".."))
logging.basicConfig(level=logging.WARNING, format="%(asctime)s %(levelname)s %(name)s: %(message)s", force=True)
logging.getLogger("src").setLevel(logging.INFO)

from src.config import load_settings
from src.transform.gold import build_gold

os.environ["FOOTBALL_DATA_API_KEY"] = dbutils.secrets.get(scope="fc", key="football_data_api_key")
settings = load_settings()

In [0]:
contagens = build_gold(spark, settings)
for tabela, linhas in contagens.items():
    print(f"{tabela}: {linhas} linha(s)")

In [0]:
%sql
SELECT encerrada, clima_disponivel, COUNT(*) AS partidas,
       MIN(data_partida) AS primeira_data, MAX(data_partida) AS ultima_data
FROM workspace.fc.gold_partidas_clima
GROUP BY ALL
ORDER BY ALL

In [0]:
%sql
SELECT * FROM workspace.fc.gold_resumo_clima
ORDER BY dimensao, faixa

In [0]:
%sql
SELECT
  mandante,
  COUNT_IF(choveu)      AS jogos_chuva,
  ROUND(100 * SUM(CASE WHEN choveu THEN pontos_mandante END) / NULLIF(3 * COUNT_IF(choveu), 0), 1)          AS aproveitamento_chuva,
  COUNT_IF(NOT choveu)  AS jogos_seco,
  ROUND(100 * SUM(CASE WHEN NOT choveu THEN pontos_mandante END) / NULLIF(3 * COUNT_IF(NOT choveu), 0), 1)  AS aproveitamento_seco
FROM workspace.fc.gold_partidas_clima
WHERE encerrada AND clima_disponivel
GROUP BY mandante
HAVING COUNT_IF(choveu) >= 3
ORDER BY aproveitamento_chuva DESC